In [43]:
#Every import
import pandas as pd
import numpy as np

In [35]:
#Loading dataset
train = pd.read_parquet("train_preprocessed.parquet")
val = pd.read_parquet("val_preprocessed.parquet")
test = pd.read_parquet("test_preprocessed.parquet")

In [45]:
#Making Quartal over Quartal changes for every feature
qoq_features = [
    'Shillers Cyclically Adjusted P/E Ratio',
    'Book/Market',
    'Enterprise Value Multiple',
    'Price/Operating Earnings (Basic, Excl. EI)',
    'Price/Operating Earnings (Diluted, Excl. EI)',
    'P/E (Diluted, Excl. EI)',
    'P/E (Diluted, Incl. EI)',
    'Price/Sales',
    'Price/Cash flow',
    'Dividend Payout Ratio',
    'Net Profit Margin',
    'Operating Profit Margin Before Depreciation',
    'Operating Profit Margin After Depreciation',
    'Gross Profit Margin',
    'Pre-tax Profit Margin',
    'Cash Flow Margin',
    'Return on Assets',
    'Return on Equity',
    'Return on Capital Employed',
    'Effective Tax Rate',
    'After-tax Return on Average Common Equity',
    'After-tax Return on Invested Capital',
    'After-tax Return on Total Stockholders Equity',
    'Pre-tax return on Net Operating Assets',
    'Pre-tax Return on Total Earning Assets',
    'Gross Profit/Total Assets',
    'Common Equity/Invested Capital',
    'Long-term Debt/Invested Capital',
    'Total Debt/Invested Capital',
    'Capitalization Ratio',
    'Interest/Average Long-term Debt',
    'Interest/Average Total Debt',
    'Cash Balance/Total Liabilities',
    'Inventory/Current Assets',
    'Receivables/Current Assets',
    'Total Long Term Debt Plus Debt in Current Liabilities/Total Assets',
    'Total Debt/EBITDA',
    'Short-Term Debt/Total Debt',
    'Current Liabilities/Total Liabilities',
    'Long-term Debt/Total Liabilities',
    'Profit Before Depreciation/Current Liabilities',
    'Operating CF/Current Liabilities',
    'Cash Flow/Total Debt',
    'Free Cash Flow/Operating Cash Flow',
    'Total Liabilities/Total Tangible Assets',
    'Long-term Debt/Book Equity',
    'Total Liabilities/Total Assets',
    'Total Debt/Capital',
    'Total Debt/Equity',
    'After-tax Interest Coverage',
    'Interest Coverage Ratio',
    'Cash Ratio',
    'Quick Ratio (Acid Test)',
    'Current Ratio',
    'Cash Conversion Cycle (Days)',
    'Inventory Turnover',
    'Asset Turnover',
    'Receivables Turnover',
    'Payables Turnover',
    'Sales/Invested Capital',
    'Sales/Stockholders Equity',
    'Sales/Working Capital',
    'Research and Development/Sales',
    'Avertising Expenses/Sales',
    'Labor Expenses/Sales',
    'Accruals/Average Assets',
    'Price/Book',
    'Trailing P/E to Growth (PEG) ratio',
    'Dividend Yield'
]

# Sorting based on ticker and date
train = train.sort_values(['Ticker','Date'])

# Making QoQ changes
for col in qoq_features:
    if col in train.columns:
        train[col + "_QoQ"] = (
            train.groupby('Ticker')[col]
            .diff()
        )


# Doing the same for validation data
val = val.sort_values(['Ticker','Date'])

for col in qoq_features:
    if col in val.columns:
        val[col + "_QoQ"] = (
            val.groupby('Ticker')[col]
            .diff()
        )


# Doing the same for testing data
test = test.sort_values(['Ticker','Date'])

for col in qoq_features:
    if col in test.columns:
        test[col + "_QoQ"] = (
            test.groupby('Ticker')[col]
            .diff()
        )

In [47]:
#Pricebasedfeatures
price_features = [

    # Market valuation multiples 
    'Shillers Cyclically Adjusted P/E Ratio',
    'Enterprise Value Multiple',
    'Price/Book',
    'Price/Sales',
    'Price/Cash flow',

    # Earnings price multiples
    'Price/Operating Earnings (Basic, Excl. EI)',
    'Price/Operating Earnings (Diluted, Excl. EI)',
    'P/E (Diluted, Excl. EI)',
    'P/E (Diluted, Incl. EI)',

    # Growth adjusted valuation
    'Trailing P/E to Growth (PEG) ratio',

    # Additional valuation signals (vaak vergeten maar sterk voor ML)
    'Book/Market',
    'Price/Book',
    'Price/Sales',

    # Dividend based valuation signals
    'Dividend Yield',
    'Dividend Payout Ratio'
]

bins_dict = {}

for col in price_features:
    if col in train.columns:
        bins_dict[col] = train[col].quantile([0.25,0.5,0.75,0.9]).values



def add_valuation_regime(df, bins_dict):

    df = df.copy()

    for col, bins in bins_dict.items():

        if col not in df.columns:
            continue

        labels = ['Low','Medium_Low','Medium_High','High','Extreme']

        df[col + "_Regime"] = pd.cut(
            df[col],
            bins=[-np.inf] + list(bins) + [np.inf],
            labels=labels
        )

    return df

train = add_valuation_regime(train, bins_dict)
val = add_valuation_regime(val, bins_dict)
test = add_valuation_regime(test, bins_dict)




In [49]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

train_ts = train.sort_values(['Ticker','Date'])

forecast_features = [
    'Net Profit Margin',
    'Cash Flow Margin'
]

for feature in forecast_features:

    rmse_list = []
    mae_list = []

    print(f"\nEvaluating {feature}")

    for ticker in train_ts['Ticker'].unique():

        ticker_data = train_ts[train_ts['Ticker']==ticker]

        series = ticker_data[feature].dropna()

        if len(series) < 20:
            continue

        train_size = int(len(series)*0.8)

        train_series = series.iloc[:train_size]
        test_series = series.iloc[train_size:]

        try:
            model = ARIMA(train_series, order=(1,1,1))
            model_fit = model.fit()

            forecast = model_fit.forecast(steps=len(test_series))

            # Metrics
            rmse = np.sqrt(mean_squared_error(test_series, forecast))
            mae = mean_absolute_error(test_series, forecast)

            rmse_list.append(rmse)
            mae_list.append(mae)

        except:
            continue




Evaluating Net Profit Margin


C:\Users\31649\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\31649\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\31649\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\31649\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\31649\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next versio


Evaluating Cash Flow Margin


C:\Users\31649\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\31649\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
C:\Users\31649\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\31649\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\31649\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: Val

In [51]:
print(f"Average RMSE {feature}:", np.mean(rmse_list))
print(f"Average MAE {feature}:", np.mean(mae_list))

Average RMSE Cash Flow Margin: 0.057490706230154155
Average MAE Cash Flow Margin: 0.05002441600862568


In [53]:
# Delete list with basic features
remove_features = [
    'Shillers Cyclically Adjusted P/E Ratio',
    'Enterprise Value Multiple',
    'Price/Book',
    'Price/Sales',
    'Price/Cash flow',
    'Price/Operating Earnings (Basic, Excl. EI)',
    'Price/Operating Earnings (Diluted, Excl. EI)',
    'P/E (Diluted, Excl. EI)',
    'P/E (Diluted, Incl. EI)',
    'Trailing P/E to Growth (PEG) ratio',
    'Book/Market',
    'Dividend Yield',
    'Dividend Payout Ratio'
]


remove_with_qoq = []

for col in train.columns:
    for feature in remove_features:
        if col == feature or col == feature + "_QoQ":
            remove_with_qoq.append(col)

# Uniek maken
remove_with_qoq = list(set(remove_with_qoq))

print("Columns to remove:")
print(remove_with_qoq)

# Verwijderen uit train, val, test
train = train.drop(columns=remove_with_qoq, errors='ignore')
val = val.drop(columns=remove_with_qoq, errors='ignore')
test = test.drop(columns=remove_with_qoq, errors='ignore')

Columns to remove:
['Price/Operating Earnings (Basic, Excl. EI)', 'Price/Sales', 'Price/Operating Earnings (Basic, Excl. EI)_QoQ', 'Shillers Cyclically Adjusted P/E Ratio', 'Enterprise Value Multiple', 'Price/Cash flow', 'Trailing P/E to Growth (PEG) ratio', 'Price/Operating Earnings (Diluted, Excl. EI)_QoQ', 'Price/Sales_QoQ', 'Book/Market_QoQ', 'Dividend Yield', 'Price/Book', 'Dividend Payout Ratio', 'Trailing P/E to Growth (PEG) ratio_QoQ', 'Dividend Yield_QoQ', 'Price/Cash flow_QoQ', 'P/E (Diluted, Incl. EI)_QoQ', 'Price/Book_QoQ', 'P/E (Diluted, Excl. EI)', 'P/E (Diluted, Excl. EI)_QoQ', 'Dividend Payout Ratio_QoQ', 'Book/Market', 'P/E (Diluted, Incl. EI)', 'Price/Operating Earnings (Diluted, Excl. EI)', 'Enterprise Value Multiple_QoQ', 'Shillers Cyclically Adjusted P/E Ratio_QoQ']


In [55]:
train.head(20)


,Ticker,EarningsDate,Close_Before,Open_After,Return,EPS_Estimate,EPS_Actual,Surprise_Pct,Year,Month,...,Price/Sales_Regime,Price/Cash flow_Regime,"Price/Operating Earnings (Basic, Excl. EI)_Regime","Price/Operating Earnings (Diluted, Excl. EI)_Regime","P/E (Diluted, Excl. EI)_Regime","P/E (Diluted, Incl. EI)_Regime",Trailing P/E to Growth (PEG) ratio_Regime,Book/Market_Regime,Dividend Yield_Regime,Dividend Payout Ratio_Regime
40,AAPL,2015-01-27,25.073328,26.077591,1.004263,0.65,0.77,17.36,2015,1,...,Medium_High,Medium_Low,Medium_Low,Medium_Low,Medium_Low,Medium_Low,Medium_Low,Medium_Low,Medium_Low,Medium_Low
41,AAPL,2015-04-27,28.995975,29.926306,0.930331,0.54,0.58,7.93,2015,4,...,Medium_High,Medium_Low,Low,Low,Low,Low,NaN,Medium_Low,Medium_Low,Medium_Low
42,AAPL,2015-07-21,29.517160,27.264312,-2.252848,0.45,0.46,1.89,2015,7,...,Medium_High,Medium_Low,Low,Low,Low,Low,NaN,Medium_Low,Medium_Low,Medium_Low
43,AAPL,2015-10-27,25.881273,26.251712,0.370439,0.47,0.49,4.65,2015,10,...,Medium_High,Low,Low,Low,Low,Low,NaN,Medium_Low,Medium_Low,Medium_Low
44,AAPL,2016-01-26,22.420633,21.654034,-0.766599,0.81,0.82,1.67,2016,1,...,Medium_Low,Low,Low,Low,Low,Low,Medium_Low,Medium_Low,Medium_High,Medium_Low
45,AAPL,2016-04-26,23.820843,21.762473,-2.058370,0.50,0.48,-5.09,2016,4,...,Medium_Low,Low,Low,Low,Low,Low,NaN,Medium_Low,Medium_High,Medium_Low
46,AAPL,2016-07-26,22.200583,23.781126,1.580543,0.35,0.36,2.74,2016,7,...,Medium_Low,Low,Low,Low,Low,Low,NaN,Medium_Low,Medium_High,Medium_Low
47,AAPL,2016-10-25,26.978102,26.212210,-0.765892,0.41,0.42,0.81,2016,10,...,Medium_High,Medium_Low,Low,Low,Low,Low,NaN,Medium_High,Medium_Low,Medium_Low
48,AAPL,2017-01-31,28.033945,29.278568,1.244623,0.81,0.84,4.28,2017,1,...,Medium_High,Medium_Low,Medium_Low,Medium_Low,Medium_Low,Medium_Low,Medium_Low,Medium_Low,Medium_Low,Medium_Low
49,AAPL,2017-05-02,33.931030,33.701866,-0.229165,0.51,0.53,3.88,2017,5,...,Medium_High,Medium_Low,Low,Low,Low,Low,NaN,Medium_Low,Medium_Low,Medium_Low


In [57]:
train.to_parquet("train_feature_engineered.parquet")
val.to_parquet("val_feature_engineered.parquet")
test.to_parquet("test_feature_engineered.parquet")